# Агент суммаризации JSON
MCP + LangChain + LangGraph + Groq. CPU достаточно. Ключ хранится только в Colab Secrets или вводится скрыто через getpass.

Этот ноутбук запускает настоящий агентный граф, а не цикл суммаризации статей. Подробности каждой функции — в README.

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path("/content/agents-homework")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/evelinashakhnazaryan/agents-homework.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed")

## Проверки без API
Включают настоящий MCP-сервер и LangGraph. Решения модели в инфраструктурном тесте заменены тестовыми сообщениями; это не реальный прогон Groq.

In [ ]:
checked = subprocess.run([sys.executable, "-m", "unittest", "-v", "test_agent"], capture_output=True, text=True)
print(checked.stdout + checked.stderr)
checked.check_returncode()

## Подключение Groq
Добавьте секрет `GROQ_API_KEY` и разрешите доступ этому ноутбуку. Если секрета нет, появится скрытое поле ввода. Значение не выводится и не сохраняется в ноутбуке.

In [ ]:
from google.colab import userdata
from getpass import getpass
try:
    api_key = userdata.get("GROQ_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    api_key = getpass("Groq API key: ")
if not api_key:
    raise ValueError("Нужен ключ Groq")
os.environ["GROQ_API_KEY"] = api_key
os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"
del api_key
print("Groq key configured (value hidden)")

## Запуск агента
Модель самостоятельно вызывает чтение → суммаризацию по ID → сохранение. Новый запуск использует уникальную папку. При исчерпании квоты запуск завершится ошибкой и оставит трассу.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
run_dir = Path("results_reproduction") / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6])
run_dir.mkdir(parents=True)
output = run_dir / "summaries.json"
completed = subprocess.run([sys.executable, "agent.py", "--input", "articles.json", "--output", str(output)], capture_output=True, text=True)
print(completed.stdout)
print(completed.stderr)
completed.check_returncode()

## Проверка и просмотр результатов
Проверяем хеш входа, все ID, число записей и непустые резюме. Затем выводим резюме для ручной оценки.

In [ ]:
import json
from agent import verify_output
result = verify_output(Path("articles.json"), output)
trace = json.loads(output.with_suffix(".trace.json").read_text(encoding="utf-8"))
assert trace["success"]
print("Articles:", result["count"], "| Model:", result["model"])
for article in result["articles"]:
    print("\n", article["id"], article.get("title", ""))
    print(article["summary"])
calls = [call["name"] for event in trace["events"] for call in event.get("tool_calls", [])]
print("\nTool calls:", calls)

## Скачать результат
Архив содержит реальные ответы и трассу; ключа в нём нет. Выполненный ноутбук можно скачать через меню «Файл → Скачать → Скачать IPYNB».

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(run_dir), "zip", run_dir)
files.download(archive)